# Training pipeline — churn prediction

Repeatable pipeline: load data → build features → split train/val/test → train baseline + candidate → evaluate → apply promotion rule → save artifacts.

Metrics: **ROC AUC** is primary (threshold-independent, reflects rank quality — the business use here is *ranking* customers by churn risk). Accuracy, precision, recall, and F1 are reported as secondary — accuracy alone is misleading on this ~26% positive-rate imbalanced target, and precision/recall map directly to business cost (false positive = wasted retention offer; false negative = an unflagged customer who churns).

In [1]:
import sys, os, json
from datetime import datetime, timezone
sys.path.append(os.path.abspath('..'))

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from features.feature_engineering import build_features, encode_for_model


## Config

In [2]:
DATA_PATH = '../data/processed/training_table.csv'
MODEL_DIR = '../models'
EVAL_DIR = '../artifacts/eval'

PROMOTION_RULE = {'min_auc': 0.80, 'max_auc_regression_vs_baseline': 0.01}


## Load data and build features

In [3]:
df = pd.read_csv(DATA_PATH)
df = df.drop_duplicates(subset='customerID')
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

df_feat = build_features(df)
y = df_feat['Churn']
X, encoders = encode_for_model(df_feat, encoders=None)
print(f'{len(X)} rows, {X.shape[1]} feature columns, positive rate {y.mean():.3f}')


6500 rows, 50 feature columns, positive rate 0.268


## Train / validation / test split (stratified 70/15/15)

In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)
print(f'train={len(X_train)}  val={len(X_val)}  test={len(X_test)}')


train=4550  val=975  test=975


## Evaluation helper

In [5]:
def evaluate(model, X, y, scaler=None):
    X_eval = scaler.transform(X) if scaler is not None else X
    proba = model.predict_proba(X_eval)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {
        'accuracy': round(accuracy_score(y, pred), 4),
        'roc_auc': round(roc_auc_score(y, proba), 4),
        'precision': round(precision_score(y, pred), 4),
        'recall': round(recall_score(y, pred), 4),
        'f1': round(f1_score(y, pred), 4),
        'n_samples': int(len(y)),
        'positive_rate': round(float(y.mean()), 4),
    }


## Baseline: Logistic Regression

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
baseline = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
baseline.fit(X_train_scaled, y_train)
baseline_val = evaluate(baseline, X_val, y_val, scaler=scaler)
baseline_val


{'accuracy': 0.7692,
 'roc_auc': 0.8579,
 'precision': 0.5476,
 'recall': 0.813,
 'f1': 0.6544,
 'n_samples': 975,
 'positive_rate': 0.2687}

## Candidate: Random Forest

In [7]:
candidate = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=10,
    class_weight='balanced', random_state=42, n_jobs=-1)
candidate.fit(X_train, y_train)
candidate_val = evaluate(candidate, X_val, y_val)
candidate_val


{'accuracy': 0.7682,
 'roc_auc': 0.8532,
 'precision': 0.5492,
 'recall': 0.7672,
 'f1': 0.6401,
 'n_samples': 975,
 'positive_rate': 0.2687}

## Promotion decision

Guardrail rule: promote the candidate only if its AUC clears the absolute floor **and** isn't meaningfully worse than the baseline.

In [8]:
promote = (
    candidate_val['roc_auc'] >= PROMOTION_RULE['min_auc']
    and candidate_val['roc_auc'] >= baseline_val['roc_auc'] - PROMOTION_RULE['max_auc_regression_vs_baseline']
)
chosen_name = 'candidate_random_forest' if promote else 'baseline_logistic_regression'
chosen_model = candidate if promote else baseline
chosen_scaler = None if promote else scaler
print(f'Promote candidate: {promote}  ->  chosen model: {chosen_name}')


Promote candidate: True  ->  chosen model: candidate_random_forest


## Test-set evaluation of the chosen model

In [9]:
test_metrics = evaluate(chosen_model, X_test, y_test, scaler=chosen_scaler)
test_metrics


{'accuracy': 0.7405,
 'roc_auc': 0.8139,
 'precision': 0.5108,
 'recall': 0.728,
 'f1': 0.6003,
 'n_samples': 975,
 'positive_rate': 0.2677}

## Save artifacts (model + evaluation report)

In [10]:
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

model_version = datetime.now(timezone.utc).strftime('v%Y%m%d_%H%M%S')
artifact = {
    'model': chosen_model, 'model_name': chosen_name,
    'scaler': chosen_scaler, 'encoders': encoders, 'model_version': model_version,
}
joblib.dump(artifact, os.path.join(MODEL_DIR, f'{model_version}.joblib'))
joblib.dump(artifact, os.path.join(MODEL_DIR, 'latest.joblib'))

report = {
    'model_version': model_version,
    'promotion_rule': PROMOTION_RULE,
    'promoted_model': chosen_name,
    'baseline_logistic_regression': {'validation': baseline_val},
    'candidate_random_forest': {'validation': candidate_val},
    'chosen_model_test_set_metrics': test_metrics,
}
with open(os.path.join(EVAL_DIR, f'eval_report_{model_version}.json'), 'w') as f:
    json.dump(report, f, indent=2)
with open(os.path.join(EVAL_DIR, 'latest_eval_report.json'), 'w') as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))


{
  "model_version": "v20260808_163110",
  "promotion_rule": {
    "min_auc": 0.8,
    "max_auc_regression_vs_baseline": 0.01
  },
  "promoted_model": "candidate_random_forest",
  "baseline_logistic_regression": {
    "validation": {
      "accuracy": 0.7692,
      "roc_auc": 0.8579,
      "precision": 0.5476,
      "recall": 0.813,
      "f1": 0.6544,
      "n_samples": 975,
      "positive_rate": 0.2687
    }
  },
  "candidate_random_forest": {
    "validation": {
      "accuracy": 0.7682,
      "roc_auc": 0.8532,
      "precision": 0.5492,
      "recall": 0.7672,
      "f1": 0.6401,
      "n_samples": 975,
      "positive_rate": 0.2687
    }
  },
  "chosen_model_test_set_metrics": {
    "accuracy": 0.7405,
    "roc_auc": 0.8139,
    "precision": 0.5108,
    "recall": 0.728,
    "f1": 0.6003,
    "n_samples": 975,
    "positive_rate": 0.2677
  }
}
